In [14]:
# Import pandas for working with tables (DataFrames)
import pandas as pd

# Folder where the data files are stored (one level up, in "data")
DATA = "../data/"

# Load the three raw World Bank files into DataFrames
# encoding="latin-1" tells pandas how to read special characters (like accents) in the files
debt    = pd.read_csv(DATA + "IDS_ALLCountries_Data.csv", encoding="latin-1")     # the main debt figures
country = pd.read_csv(DATA + "IDS_CountryMetaData.csv", encoding="latin-1")       # details about each country
series  = pd.read_csv(DATA + "IDS_SeriesMetaData.csv", encoding="latin-1")        # details about each indicator

# Print the size (rows, columns) of the raw debt table as a starting point
print("Start:", debt.shape)

Start: (62983, 39)


In [15]:
# Show 10 randomly chosen values from the "Country Name" column
debt["Country Name"].sample(10)

40955                Morocco
59983    Upper middle income
52232              St. Lucia
51822              Sri Lanka
39255                Moldova
62611               Zimbabwe
45555               Paraguay
43765                Nigeria
13000          Cote d'Ivoire
39583                Moldova
Name: Country Name, dtype: str

In [16]:
# Remove rows where "Country Code" is empty (these are junk/blank rows)
debt = debt.dropna(subset=["Country Code"])

# Print the new size (rows, columns) to see how many rows were removed
print("After removing junk rows:", debt.shape)

After removing junk rows: (62978, 39)


In [20]:
# Remove extra spaces from the start and end of each country code
debt["Country Code"] = debt["Country Code"].str.strip()

# Do the same for each country name
debt["Country Name"] = debt["Country Name"].str.strip()

# Show the first country code exactly as stored (quotes reveal any hidden spaces)
print(repr(debt["Country Code"].iloc[0]))

'AFG'


In [21]:
# Keep only the rows in the country table that have a Region (real countries, not aggregates)
real_countries = country[country["Region"].notna()]

# Keep only the debt rows whose country code is in that list of real countries
debt = debt[debt["Country Code"].isin(real_countries["Code"])]

# Print how many different countries remain
print("Countries left:", debt["Country Code"].nunique())

# Print the new size (rows, columns) of the debt table
print("Rows left:", debt.shape)

Countries left: 120
Rows left: (55049, 39)


In [22]:
# Remove the two "Counterpart-Area" columns from the debt table
debt = debt.drop(columns=["Counterpart-Area Name", "Counterpart-Area Code"])

# Print the first 6 column names to confirm the columns were removed
print(debt.columns.tolist()[:6])

['Country Name', 'Country Code', 'Series Name', 'Series Code', '2000', '2001']


In [23]:
# Find all column names that are made only of digits (these are the year columns)
year_cols = [c for c in debt.columns if c.isdigit()]   # '2000' ... '2032'

# Reshape the table from wide (one column per year) to long (one row per year)
debt_long = debt.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],   # columns to keep as they are
    value_vars=year_cols,                                                     # columns to turn into rows
    var_name="year",                                                          # name of the new column holding the year
    value_name="value",                                                       # name of the new column holding the number
)

# Convert the year from text (like "2000") to a whole number (2000)
debt_long["year"] = debt_long["year"].astype(int)

# Print the size (rows, columns) of the long table
print("Long format:", debt_long.shape)

# Show the first 5 rows
debt_long.head()

Long format: (1816617, 6)


,Country Name,Country Code,Series Name,Series Code,year,value
0,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.DPPG,2000,NaN
1,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.OFFT,2000,NaN
2,Afghanistan,AFG,Average grace period on new external debt comm...,DT.GPA.PRVT,2000,NaN
3,Afghanistan,AFG,Average grant element on new external debt com...,DT.GRE.DPPG,2000,NaN
4,Afghanistan,AFG,Average grant element on new external debt com...,DT.GRE.OFFT,2000,NaN


In [24]:
# Show a technical summary of the debt_long table
debt_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 1816617 entries, 0 to 1816616
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Country Name  str    
 1   Country Code  str    
 2   Series Name   str    
 3   Series Code   str    
 4   year          int64  
 5   value         float64
dtypes: float64(1), int64(1), str(4)
memory usage: 213.8 MB


In [25]:
# Remove rows where "value" is empty (years with no data)
debt_long = debt_long.dropna(subset=["value"])

# Print the new size (rows, columns) after removing the empty rows
print("After dropping empty values:", debt_long.shape)

# Keep only rows from 2024 or earlier (drops future years such as 2025 to 2032)
debt_long = debt_long[debt_long["year"] <= 2024]

# Print the size again after the year filter
print("After keeping 2000-2024:", debt_long.shape)

After dropping empty values: (1142836, 6)
After keeping 2000-2024: (1060074, 6)


In [26]:
# Remember how many rows there are before removing duplicates
before = len(debt_long)

# Drop rows that repeat the same country, indicator and year (keeps the first one)
debt_long = debt_long.drop_duplicates(subset=["Country Code", "Series Code", "year"])

# Print how many rows were removed
print("Duplicates removed:", before - len(debt_long))

Duplicates removed: 0


In [27]:
# 1) Countries table: build a clean table of real countries
# Pick 4 columns from the real-country list and give them cleaner names
countries = (real_countries[["Code", "Long Name", "Region", "Income Group"]]
             .rename(columns={"Code": "country_code", "Long Name": "country_name",
                              "Region": "region", "Income Group": "income_group"}))
# Replace missing income groups with the label "Unclassified"
countries["income_group"] = countries["income_group"].fillna("Unclassified")

# 2) Indicators table: take names from the debt data itself, then add the topic
# Get each unique indicator code + name pair, and rename the columns
indicators = (debt_long[["Series Code", "Series Name"]]
              .drop_duplicates()
              .rename(columns={"Series Code": "indicator_code",
                               "Series Name": "indicator_name"}))

# Prepare a small lookup table of indicator code -> topic from the series metadata
topics = series[["Code", "Topic"]].rename(columns={"Code": "indicator_code",
                                                    "Topic": "topic"})
# Attach the topic to each indicator (left join keeps all indicators, even without a topic)
indicators = indicators.merge(topics, on="indicator_code", how="left")
# Replace missing topics with the label "Unknown"
indicators["topic"] = indicators["topic"].fillna("Unknown")

# Flag indicators measured in US dollars (needed for the debt questions later)
# 1 if the name contains "current US$", otherwise 0
indicators["is_usd"] = indicators["indicator_name"].str.contains(
    "current US$", regex=False).astype(int)

# 3) Debt data table (the main table)
# Keep only the 4 needed columns and rename them to match the other tables
debt_data = (debt_long[["Country Code", "Series Code", "year", "value"]]
             .rename(columns={"Country Code": "country_code",
                              "Series Code": "indicator_code"}))

# Print the number of rows in each of the three tables
print(len(countries), len(indicators), len(debt_data))

120 576 1060074


In [28]:
# Check that each indicator code appears only once (True = good, no duplicates)
print("Unique indicator codes:", indicators["indicator_code"].is_unique)

# Count how many indicators are measured in US dollars (adds up the 1s in is_usd)
print("US$ indicators:", indicators["is_usd"].sum())

# Show the earliest and latest year in the debt data
print("Years:", debt_data["year"].min(), "to", debt_data["year"].max())

# Count how many different countries appear in the debt data
print("Countries in debt_data:", debt_data["country_code"].nunique())

# Show how many missing values each column has ("\n" starts the table on a new line)
print("Missing values left:\n", debt_data.isna().sum())

Unique indicator codes: True
US$ indicators: 538
Years: 2000 to 2024
Countries in debt_data: 120
Missing values left:
 country_code      0
indicator_code    0
year              0
value             0
dtype: int64


In [29]:
# Save the cleaned countries table to a CSV file (index=False leaves out the row numbers)
countries.to_csv(DATA + "clean_countries.csv", index=False)

# Save the cleaned indicators table
indicators.to_csv(DATA + "clean_indicators.csv", index=False)

# Save the cleaned debt data table (the main table)
debt_data.to_csv(DATA + "clean_debt_data.csv", index=False)

# Confirm that all three files were written
print("Saved.")

Saved.
